In [6]:
import numpy as np
import pandas as pd
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold
from joblib import Parallel, delayed, dump, load

In [7]:
df_train = pd.read_csv("Data/encode_train.csv")
df_valid = pd.read_csv("Data/encode_valid.csv")
df_test = pd.read_csv("Data/encode_test.csv")

In [8]:
bool_cols = [
    'FirstTimeHomebuyerFlag',
    'SuperConformingFlag',
    'CreditScore_MissFLag',
    'OriginalDTI_MissFLag',
    'HighRiskCredit',
    'HighLTV',
    'HighDTI',
    'HighInterestRate',
    'LTV_MissFlag'
]

cat_cols = [
    'NumberOfUnits',
    'OccupancyStatus',
    'Channel',
    'PropertyType',
    'LoanPurpose',
    'ProgramIndicator',
    'PropertyValMethod',
    'BalloonIndicator'
]

num_cols = [
    'CreditScore',
    'MI_Pct',
    'OriginalDTI',
    'OriginalUPB',
    'OriginalLTV',
    'OriginalInterestRate',
    'OriginalLoanTerm',
    'NumberOfBorrowers',
    '0_1_UPB_Diff',
    '1_2_UPB_Diff',
    '2_3_UPB_Diff',
    '3_4_UPB_Diff',
    '4_5_UPB_Diff',
    '5_6_UPB_Diff',
    '6_7_UPB_Diff',
    '7_8_UPB_Diff',
    '8_9_UPB_Diff',
    '9_10_UPB_Diff',
    '10_11_UPB_Diff',
    '11_12_UPB_Diff',
    '12_13_UPB_Diff',
    'longest_unchanged_UPB',
    'avg_repayment_ratio',
    'pct_months_late',
    'Principal_reduction_rate',
    'Amortization_slope',
    'UPB_rebound_count',
    'UPB_autocorr',
    'UPB_skew',
    'UPB_kurtosis',
    'LTV_mean',
    'LTV_std',
    'LTV_slope',
    'LTV_rebound_count',
    'Interest_rate_std',
    'Interest_rate_slope',
    'Early_repayment_ratio',
    'CreditScore_LTV_Ratio',
    'CreditScore_DTI_Ratio',
    'LTV_DTI_Product',
    'DebtServiceRatio',
    'CompositeRiskScore',
    'OriginalLTV_delta',
    'MSA_freq_enc',
    'PropertyState_freq_enc',
    'SellerName_freq_enc',
    'ServicerName_freq_enc'
]

In [9]:
features = bool_cols + cat_cols + num_cols

X_train = df_train[features]
X_valid = df_valid[features]
X_test = df_test[features]

y_valid = df_valid[["index", "target"]]

In [10]:
def scale_scores_to_unit(anomaly_scores):
    min_score = np.min(anomaly_scores)
    max_score = np.max(anomaly_scores)
    scaled_scores = (anomaly_scores - min_score) / (max_score - min_score + 1e-12)
    return scaled_scores

In [11]:
def adjusted_gower_normalization(abs_errors, X_train_df, feat, num_cols, cat_cols, bool_cols, alpha=0.05, eps=1e-8):
    if feat in num_cols:
        feature_vals = X_train_df[feat].values
        q_low = np.quantile(feature_vals, alpha)
        q_high = np.quantile(feature_vals, 1 - alpha)
        scale = max(q_high - q_low, eps)
        return np.abs(abs_errors) / scale
    elif feat in cat_cols or feat in bool_cols:
        return abs_errors
    else:
        return abs_errors

class RFOD:
    def __init__(self,
                 n_estimators=200,
                 n_splits=5,
                 feature_frac=0.7,
                 patience=5,
                 n_jobs=-1,
                 agg_weight_method="adaptive",
                 random_state=42,
                 verbose=True):
        self.n_estimators = n_estimators
        self.n_splits = n_splits
        self.feature_frac = feature_frac
        self.patience = patience
        self.n_jobs = n_jobs
        self.agg_weight_method = agg_weight_method
        self.random_state = random_state
        self.verbose = verbose

        self.selected_features_ = None
        self.feature_ranking_ = None
        self.preprocessor_ = None
        self.feature_indices_ = None
        self.final_AP_ = None
        self.ROC_AUC_ = None

        self.feature_models_ = {}

        self.feature_model_inputs_ = {}
        self.X_train_original = None

    def _weight_from_uncertainty(self, uncert, residual_mean=None, eps=1e-8):
        method = self.agg_weight_method
        if method == "inv":
            return 1.0 / (uncert + eps)
        elif method == "logodds":
            u = np.clip(uncert, eps, 1.0 - eps)
            return np.log((1.0 - u) / (u + eps))
        elif method == "adaptive":
            if residual_mean is None:
                residual_mean = np.mean(uncert, axis=0)
            return 1.0 / (uncert + 0.5 * residual_mean + eps)
        return np.ones_like(uncert)

    def _train_one_feature(self, i, feat, Xtr_proc, Xval_proc, feature_indices, all_features,
                           num_cols, cat_cols, bool_cols, y_valid, X_train_df):

        rng = np.random.default_rng(self.random_state + i)
        others = [f for f in all_features if f != feat]
        selected_features = rng.choice(others, size=max(1, int(len(others) * self.feature_frac)), replace=False)
        idxs_sel = [feature_indices[f] for f in selected_features]

        Xtr_sel = Xtr_proc[:, idxs_sel]
        Xval_sel = Xval_proc[:, idxs_sel]
        ytr_full = Xtr_proc[:, feature_indices[feat]]
        yval_full = Xval_proc[:, feature_indices[feat]]

        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        preds_val = np.zeros(len(yval_full))
        uncert_val = np.zeros(len(yval_full))

        model_full = None

        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(Xtr_sel)):
            Xtr, Xval = Xtr_sel[tr_idx], Xtr_sel[val_idx]
            ytr = ytr_full[tr_idx]

            if feat in cat_cols or feat in bool_cols:
                model = RandomForestClassifier(n_estimators=self.n_estimators,
                                               random_state=self.random_state + fold_idx,
                                               n_jobs=1)
                ytr_ = np.round(ytr).astype(int)
                model.fit(Xtr, ytr_)
                prob = model.predict_proba(Xval_sel)
                prob_max = np.max(prob, axis=1)
                prob_true = np.array([
                    prob[i, np.where(model.classes_ == np.round(yval_full[i]).astype(int))[0][0]]
                    if np.round(yval_full[i]).astype(int) in model.classes_ else 0
                    for i in range(len(yval_full))
                ])
                score_fold = 1.0 - prob_true
                uncert_fold = 1.0 - prob_max
            else:
                model = RandomForestRegressor(n_estimators=self.n_estimators,
                                              max_depth=6,
                                              random_state=self.random_state + fold_idx,
                                              n_jobs=1)
                model.fit(Xtr, ytr)
                all_preds = np.vstack([t.predict(Xval_sel) for t in model.estimators_])
                mean_pred = np.mean(all_preds, axis=0)
                std_pred = np.std(all_preds, axis=0)
                abs_err = np.abs(yval_full - mean_pred)
                score_fold = adjusted_gower_normalization(abs_err, X_train_df, feat, num_cols, cat_cols, bool_cols)
                uncert_fold = std_pred

            preds_val += score_fold / self.n_splits
            uncert_val += uncert_fold / self.n_splits

        if feat in cat_cols or feat in bool_cols:
            model_full = RandomForestClassifier(n_estimators=self.n_estimators,
                                                random_state=self.random_state,
                                                n_jobs=1)
            y_full_ = np.round(ytr_full).astype(int)
            model_full.fit(Xtr_sel, y_full_)
        else:
            model_full = RandomForestRegressor(n_estimators=self.n_estimators,
                                               max_depth=8,
                                               random_state=self.random_state,
                                               n_jobs=1)
            model_full.fit(Xtr_sel, ytr_full)

        ap_single = average_precision_score(y_valid, preds_val)

        return i, feat, preds_val, uncert_val, ap_single, model_full, selected_features

    def fit(self, X_train, X_valid, y_valid, num_cols, cat_cols, bool_cols):
        start_time = time.time()
        np.random.seed(self.random_state)

        self.X_train_original = X_train.copy()
        all_features = num_cols + cat_cols + bool_cols
        self.feature_indices_ = {f: i for i, f in enumerate(all_features)}

        self.preprocessor_ = ColumnTransformer([
            ("num", RobustScaler(), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
            ("bool", "passthrough", bool_cols)
        ])

        Xtr_proc = self.preprocessor_.fit_transform(X_train)
        Xval_proc = self.preprocessor_.transform(X_valid)

        if self.verbose:
            print(f"\nStep 1: RFOD with CV, feature bagging, and Gower normalization ({len(all_features)} features):\n")

        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self._train_one_feature)(
                i, feat, Xtr_proc, Xval_proc, self.feature_indices_, all_features,
                num_cols, cat_cols, bool_cols, y_valid, X_train
            )
            for i, feat in enumerate(all_features)
        )

        if self.verbose:
            print(f"Training completed in {(time.time() - start_time) / 60:.2f} minutes.\n")

        results.sort(key=lambda x: x[0])
        cell_scores = np.zeros((Xval_proc.shape[0], len(all_features)))
        cell_uncert = np.zeros((Xval_proc.shape[0], len(all_features)))
        feature_ap = []

        # Store models for inference
        self.feature_models_ = {}
        self.feature_model_inputs_ = {}

        for i, feat, score, uncert, ap_single, model_full, selected_features in results:
            cell_scores[:, i] = score
            cell_uncert[:, i] = uncert
            feature_ap.append((feat, ap_single))
            self.feature_models_[feat] = model_full
            self.feature_model_inputs_[feat] = selected_features

        feature_ap = sorted(feature_ap, key=lambda x: x[1], reverse=True)
        ranked_feats = [f for f, _ in feature_ap]

        if self.verbose:
            print("\nGreedy subset feature selection:\n")
        selected, best_ap, no_improve = [], 0.0, 0
        progress = []

        for feat in ranked_feats:
            temp_feats = selected + [feat]
            idxs = [self.feature_indices_[f] for f in temp_feats]
            weights = self._weight_from_uncertainty(cell_uncert[:, idxs], residual_mean=np.mean(cell_scores[:, idxs], axis=0))
            combined_score = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
            ap = average_precision_score(y_valid, combined_score)
            progress.append((feat, ap))
            if ap > best_ap:
                best_ap = ap
                selected.append(feat)
                no_improve = 0
                if self.verbose:
                    print(f"Added: {feat:25s} | AP = {ap:.4f}")
            else:
                no_improve += 1
                if self.verbose:
                    print(f"Rejected: {feat:25s} | AP = {ap:.4f}")
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"\nEarly stopping (no improvement for {self.patience} rounds)\n")
                break

        idxs = [self.feature_indices_[f] for f in selected]
        weights = self._weight_from_uncertainty(cell_uncert[:, idxs])
        final_combined = np.sum(weights * cell_scores[:, idxs], axis=1) / (np.sum(weights, axis=1) + 1e-8)
        self.final_AP_ = average_precision_score(y_valid, final_combined)
        self.ROC_AUC_ = roc_auc_score(y_valid, final_combined)
        self.selected_features_ = selected
        self.feature_ranking_ = pd.DataFrame(feature_ap, columns=["Feature", "SingleFeature_AP"])

        if self.verbose:
            print(f"\nFinal RFOD AP = {self.final_AP_:.4f}, ROC-AUC = {self.ROC_AUC_:.4f}")
            print("Selected Features:", selected)

        return self

    def predict(self, X_new):
        X_new_proc = self.preprocessor_.transform(X_new)
        n_samples = X_new_proc.shape[0]
        n_features = len(self.selected_features_)

        cell_scores = np.zeros((n_samples, n_features))
        cell_uncert = np.zeros((n_samples, n_features))

        for i, feat in enumerate(self.selected_features_):
            model = self.feature_models_[feat]
            idx_feat = self.feature_indices_[feat]

            selected_features_input = self.feature_model_inputs_[feat]
            input_idxs = [self.feature_indices_[f] for f in selected_features_input]

            input_idxs = np.array(input_idxs, dtype=int)

            X_input = X_new_proc[:, input_idxs]

            y_true = X_new_proc[:, idx_feat]

            all_preds = np.vstack([t.predict(X_input) for t in model.estimators_])
            mean_pred = np.mean(all_preds, axis=0)
            std_pred = np.std(all_preds, axis=0)

            if feat in self.feature_indices_ and feat in self.X_train_original.columns and feat in self.selected_features_:
                agd_score = adjusted_gower_normalization(
                    mean_pred - y_true,
                    self.X_train_original,
                    feat,
                    self.selected_features_, # restrict to working feature list
                    cat_cols,
                    bool_cols)
            elif feat in cat_cols or feat in bool_cols:
                if hasattr(model, "predict_proba"):
                    proba = model.predict_proba(X_input)
                    classes = model.classes_
                    prob_true = np.array([
                        proba[j, np.where(classes == y_true[j])[0][0]] if y_true[j] in classes else 0
                        for j in range(n_samples)
                    ])
                    agd_score = 1.0 - prob_true
                else:
                    agd_score = np.abs(mean_pred - y_true)
            else:
                agd_score = np.abs(mean_pred - y_true)

            cell_scores[:, i] = agd_score
            cell_uncert[:, i] = std_pred

        weights = self._weight_from_uncertainty(cell_uncert)
        weighted_scores = np.sum(weights * cell_scores, axis=1) / (np.sum(weights, axis=1) + 1e-8)
        return weighted_scores

    def save(self, path="rfod_model.joblib"):
        dump(self, path)
        if self.verbose:
            print(f"Model saved to {path}")

    @classmethod
    def load(cls, path="rfod_model.joblib"):
        print(f"Loading model from {path}")
        return load(path)


In [12]:
rfod1 = RFOD(n_estimators=300, n_splits=8, feature_frac=0.8, patience=10, random_state=42)

rfod1.fit(X_train, X_valid, y_valid["target"], num_cols, cat_cols, bool_cols)


Step 1: RFOD with CV, feature bagging, and Gower normalization (64 features):



/Users/aviralgoyal/Desktop/NUS/Courses/CS5344_Big_Data_Technology/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training completed in 144.77 minutes.


Greedy subset feature selection:

Added: pct_months_late           | AP = 0.4293
Added: HighInterestRate          | AP = 0.4769
Added: Early_repayment_ratio     | AP = 0.4854
Rejected: UPB_skew                  | AP = 0.4840
Rejected: HighRiskCredit            | AP = 0.4135
Rejected: longest_unchanged_UPB     | AP = 0.4796
Rejected: UPB_autocorr              | AP = 0.4717
Rejected: HighDTI                   | AP = 0.4732
Rejected: NumberOfUnits             | AP = 0.4743
Rejected: CreditScore_MissFLag      | AP = 0.3705
Rejected: UPB_rebound_count         | AP = 0.4624
Added: DebtServiceRatio          | AP = 0.4856
Rejected: UPB_kurtosis              | AP = 0.4796
Rejected: OriginalDTI               | AP = 0.4749
Rejected: HighLTV                   | AP = 0.3439
Rejected: 11_12_UPB_Diff            | AP = 0.4829
Rejected: 12_13_UPB_Diff            | AP = 0.4825
Rejected: LTV_MissFlag              | AP = 0.3852
Rejected: 9_10_UPB_Diff             | 

In [13]:
rfod1.save('rfod_model1.joblib')

Model saved to rfod_model1.joblib


In [14]:
rfod_1 = rfod1.load('rfod_model1.joblib')

Loading model from rfod_model1.joblib


In [15]:
anom = rfod_1.predict(X_valid)
anom_scaled = scale_scores_to_unit(anom)
ap = average_precision_score(y_valid['target'], anom_scaled)

print("Validation AP using RFOD:", ap)

Validation AP using RFOD: 0.45559578576677673


In [16]:
anom_scores = (rfod_1.predict(X_test))
anom_scores_scaled = scale_scores_to_unit(anom_scores)

sub = pd.DataFrame()

sub['Id'] = df_test['Id']
sub['target'] = anom_scores_scaled

sub.to_csv('Data/sub_rfod1.csv', index=False)

In [17]:
rfod2 = RFOD(n_estimators=400, n_splits=7, feature_frac=0.65, patience=12, random_state=42)

rfod2.fit(X_train, X_valid, y_valid["target"], num_cols, cat_cols, bool_cols)


Step 1: RFOD with CV, feature bagging, and Gower normalization (64 features):



/Users/aviralgoyal/Desktop/NUS/Courses/CS5344_Big_Data_Technology/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Training completed in 138.21 minutes.


Greedy subset feature selection:

Added: HighInterestRate          | AP = 0.4089
Added: pct_months_late           | AP = 0.4461
Added: Early_repayment_ratio     | AP = 0.4562
Added: longest_unchanged_UPB     | AP = 0.4583
Rejected: HighRiskCredit            | AP = 0.4076
Rejected: UPB_autocorr              | AP = 0.4502
Added: HighDTI                   | AP = 0.4629
Rejected: UPB_kurtosis              | AP = 0.4574
Rejected: NumberOfUnits             | AP = 0.4550
Rejected: UPB_skew                  | AP = 0.4581
Rejected: OriginalUPB               | AP = 0.4569
Rejected: CreditScore_MissFLag      | AP = 0.3423
Rejected: HighLTV                   | AP = 0.3547
Added: OriginalDTI               | AP = 0.4670
Rejected: 12_13_UPB_Diff            | AP = 0.4636
Rejected: LTV_MissFlag              | AP = 0.3985
Rejected: 9_10_UPB_Diff             | AP = 0.4572
Rejected: 11_12_UPB_Diff            | AP = 0.4637
Rejected: BalloonIndicator          | AP = 0

In [18]:
rfod2.save('rfod_model2.joblib')

Model saved to rfod_model2.joblib


In [19]:
rfod_2 = rfod2.load('rfod_model2.joblib')

Loading model from rfod_model2.joblib


In [20]:
anomaly = rfod_2.predict(X_valid)
anomaly_scaled = scale_scores_to_unit(anomaly)
ap = average_precision_score(y_valid['target'], anomaly_scaled)

print("Validation AP using RFOD:", ap)

Validation AP using RFOD: 0.4260243955071611


In [21]:
anom_scores = (rfod_2.predict(X_test))
anom_scores_scaled = scale_scores_to_unit(anom_scores)

sub = pd.DataFrame()

sub['Id'] = df_test['Id']
sub['target'] = anom_scores_scaled

sub.to_csv('Data/sub_rfod2.csv', index=False)

In [22]:
anom1_val = rfod_1.predict(X_valid)
anom2_val = rfod_2.predict(X_valid)

ensemble_valid = 0.5 * anom1_val + 0.5 * anom2_val
ensemble_valid_scaled = scale_scores_to_unit(ensemble_valid)

print("Ensemble AP:", average_precision_score(y_valid['target'], ensemble_valid_scaled))

Ensemble AP: 0.4817286644170019


In [23]:
anom1_test = rfod_1.predict(X_test)
anom2_test = rfod_2.predict(X_test)

ensemble_test = 0.5 * anom1_test + 0.5 * anom2_test
ensemble_test_scaled = scale_scores_to_unit(ensemble_test)

sub = pd.DataFrame()

sub['Id'] = df_test['Id']
sub['target'] = ensemble_test_scaled

sub.to_csv('Data/sub_ensemble.csv', index=False)